<a href="https://colab.research.google.com/github/thamiraaa/RAG-Customer-Support/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import os

# Paste your copied Kaggle API content between the triple quotes below
kaggle_api_content = '''
{"username":"thamirahaids","key":"KGAT_4b81a34a5dc2053f9dc3e45118f05937"}
'''

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(kaggle_api_content.strip())

os.chmod('/root/.kaggle/kaggle.json', 0o600)

print("kaggle.json created successfully")

kaggle.json created successfully


In [ ]:
!pip install -q kaggle
!kaggle datasets download -d thoughtvector/customer-support-on-twitter
!unzip -q customer-support-on-twitter.zip

Dataset URL: https://www.kaggle.com/datasets/thoughtvector/customer-support-on-twitter
License(s): CC-BY-NC-SA-4.0
100% 169M/169M [00:02<00:00, 72.2MB/s]



In [ ]:
import pandas as pd

df = pd.read_csv("twcs.csv")

print("Total rows:", len(df))
print("Columns:", list(df.columns))

FileNotFoundError: [Errno 2] No such file or directory: 'twcs.csv'

In [ ]:
import os
print(os.listdir())

['.config', 'sample.csv', 'twcs', 'customer-support-on-twitter.zip', 'sample_data']


In [ ]:
import os
print(os.listdir("twcs"))





['twcs.csv']


In [ ]:
import pandas as pd

df = pd.read_csv("twcs/twcs.csv")

print("Total rows:", len(df))
print("Columns:", list(df.columns))

Total rows: 2811774
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


In [ ]:
# 1. Quick look at the top few rows
print("--- FIRST 5 ROWS ---")
print(df.head())

# 2. Check inbound (True = customer message, False = support agent/brand message)
print("\n--- INBOUND VALUE COUNTS ---")
print(df['inbound'].value_counts())

# 3. Check unique authors/brands
print("\n--- TOP AUTHORS ---")
print(df['author_id'].value_counts().head(10))

--- FIRST 5 ROWS ---
   tweet_id   author_id  inbound                      created_at  \
0         1  sprintcare    False  Tue Oct 31 22:10:47 +0000 2017   
1         2      115712     True  Tue Oct 31 22:11:45 +0000 2017   
2         3      115712     True  Tue Oct 31 22:08:27 +0000 2017   
3         4  sprintcare    False  Tue Oct 31 21:54:49 +0000 2017   
4         5      115712     True  Tue Oct 31 21:49:35 +0000 2017   

                                                text response_tweet_id  \
0  @115712 I understand. I would like to assist y...                 2   
1      @sprintcare and how do you propose we do that               NaN   
2  @sprintcare I have sent several private messag...                 1   
3  @115712 Please send us a Private Message so th...                 3   
4                                 @sprintcare I did.                 4   

   in_response_to_tweet_id  
0                      3.0  
1                      1.0  
2                      4.0  
3        

In [ ]:
# Separate inbound (customer) and outbound (brand) tweets
first_inbound = df[df['inbound'] == True]
outbound = df[df['inbound'] == False]

# Merge customer tweets with the brand tweets that responded to them
qa_pairs = pd.merge(
    first_inbound,
    outbound,
    left_on='tweet_id',
    right_on='in_response_to_tweet_id',
    suffixes=('_customer', '_brand')
)

# Keep relevant columns and clean up text
qa_pairs = qa_pairs[['tweet_id_customer', 'author_id_customer', 'text_customer',
                     'author_id_brand', 'text_brand', 'created_at_customer']]

print(f"Total QA Pairs linked: {len(qa_pairs):,}")
print("\n--- SAMPLE QA PAIR ---")
print("Customer:", qa_pairs.iloc[0]['text_customer'])
print("Brand (", qa_pairs.iloc[0]['author_id_brand'], "):", qa_pairs.iloc[0]['text_brand'])

Total QA Pairs linked: 1,261,888

--- SAMPLE QA PAIR ---
Customer: @sprintcare I have sent several private messages and no one is responding as usual
Brand ( sprintcare ): @115712 I understand. I would like to assist you. We would need to get you into a private secured link to further assist.


In [ ]:
import re

# 1. Choose a high-volume brand (e.g., AmazonHelp)
brand_name = 'AmazonHelp'
brand_qa = qa_pairs[qa_pairs['author_id_brand'] == brand_name].copy()

# 2. Function to clean twitter handles and extra whitespaces
def clean_tweet(text):
    text = re.sub(r'@[A-Za-z0-9_]+', '', str(text))  # Remove @mentions
    text = re.sub(r'\s+', ' ', text).strip()         # Remove extra spaces
    return text

brand_qa['clean_customer'] = brand_qa['text_customer'].apply(clean_tweet)
brand_qa['clean_brand'] = brand_qa['text_brand'].apply(clean_tweet)

# 3. Take a clean sample for processing (e.g., 5,000 pairs)
sample_df = brand_qa[['clean_customer', 'clean_brand']].dropna().head(5000)

print(f"Total QA pairs for {brand_name}: {len(brand_qa):,}")
print(f"Sample size selected: {len(sample_df)}")
print("\n--- CLEANED SAMPLE ---")
print("Customer:", sample_df.iloc[0]['clean_customer'])
print("Brand:", sample_df.iloc[0]['clean_brand'])

Total QA pairs for AmazonHelp: 168,814
Sample size selected: 5000

--- CLEANED SAMPLE ---
Customer: 電話で対応してもらいましたが改良されませんでした。 保証期間も過ぎてるので買い直しになるんでしょうね。
Brand: カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET


In [ ]:
# Install langdetect if not available (run without ! if not using Colab magic)
!pip install -q langdetect

from langdetect import detect

# Filter for English queries
def is_english(text):
    try:
        return detect(text) == 'en'
    except:
        return False

# Filter 5,000 English QA pairs
brand_qa['is_en'] = brand_qa['clean_customer'].apply(is_english)
english_sample = brand_qa[brand_qa['is_en']][['clean_customer', 'clean_brand']].head(5000)

print(f"Total English QA pairs sampled: {len(english_sample)}")
print("\n--- ENGLISH QA SAMPLE ---")
print("Customer:", english_sample.iloc[0]['clean_customer'])
print("Brand:", english_sample.iloc[0]['clean_brand'])

# Save to CSV for the next RAG/Retrieval steps
english_sample.to_csv("amazon_help_sample.csv", index=False)
print("\nSaved sample to 'amazon_help_sample.csv'")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Total English QA pairs sampled: 5000

--- ENGLISH QA SAMPLE ---
Customer: 3 different people have given 3 different answers and I still don't have my order. Says delivered Saturday, was not, I was home all day
Brand: We'd like to take a further look into this with you! Please reach us by phone or chat here: https://t.co/hApLpMlfHN ^AG

Saved sample to 'amazon_help_sample.csv'


In [ ]:
from langdetect import detect

# Function to safely detect language
def is_english(text):
    try:
        return detect(text) == 'en'
    except:
        return False

# Filter for 5,000 English QA pairs
brand_qa['is_en'] = brand_qa['clean_customer'].apply(is_english)
english_sample = brand_qa[brand_qa['is_en']][['clean_customer', 'clean_brand']].head(5000)

print(f"Total English QA pairs sampled: {len(english_sample)}")
print("\n--- ENGLISH QA SAMPLE ---")
print("Customer:", english_sample.iloc[0]['clean_customer'])
print("Brand:", english_sample.iloc[0]['clean_brand'])

# Save to CSV for vector indexing & RAG steps
english_sample.to_csv("amazon_help_sample.csv", index=False)
print("\nSaved sample to 'amazon_help_sample.csv'")

Total English QA pairs sampled: 5000

--- ENGLISH QA SAMPLE ---
Customer: 3 different people have given 3 different answers and I still don't have my order. Says delivered Saturday, was not, I was home all day
Brand: We'd like to take a further look into this with you! Please reach us by phone or chat here: https://t.co/hApLpMlfHN ^AG

Saved sample to 'amazon_help_sample.csv'


In [ ]:
# 1. Install required packages
!pip install -q sentence-transformers faiss-cpu

import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 2. Load our cleaned sample
df_sample = pd.read_csv("amazon_help_sample.csv")

# 3. Load lightweight, high-performance embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Generating embeddings for customer queries...")
customer_queries = df_sample['clean_customer'].tolist()
embeddings = embedding_model.encode(customer_queries, show_progress_bar=True)

# 4. Initialize FAISS Vector Index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

print(f"\nVector Index successfully built! Total indexed documents: {index.ntotal}")

# 5. Define a retrieval test function
def retrieve_similar_qa(query, k=2):
    query_vector = embedding_model.encode([query])
    distances, indices = index.search(np.array(query_vector).astype('float32'), k)

    results = []
    for idx in indices[0]:
        results.append({
            "Past Customer Query": df_sample.iloc[idx]['clean_customer'],
            "Amazon Response": df_sample.iloc[idx]['clean_brand']
        })
    return results

# Test retrieval on a sample query
test_query = "My package says delivered but I never received it"
print(f"\n--- TESTING RETRIEVAL FOR: '{test_query}' ---")
for i, res in enumerate(retrieve_similar_qa(test_query, k=2), 1):
    print(f"\nMatch {i}:")
    print("Past Query:", res["Past Customer Query"])
    print("Past Answer:", res["Amazon Response"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 20.7 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings for customer queries...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]


Vector Index successfully built! Total indexed documents: 5000

--- TESTING RETRIEVAL FOR: 'My package says delivered but I never received it' ---

Match 1:
Past Query: It said delivered but I haven't gotten it,
Past Answer: Oh no! I'm so sorry your package did not arrive! Check out the following Help page for more insight on packages scanned as delivered, https://t.co/cRngkypMPA. ^AV

Match 2:
Past Query: Had a package scheduled for delivery today and it didn’t come I’m not happy
Past Answer: I'm sorry your shipment hasn't arrived; what is the latest status on the order: https://t.co/Y5jpI9gRhE? ^LR


In [ ]:
# 1. Define RAG Prompt Template
def generate_rag_prompt(user_query, retrieved_matches):
    context_str = ""
    for i, match in enumerate(retrieved_matches, 1):
        context_str += f"Context {i}:\nCustomer Query: {match['Past Customer Query']}\nBrand Resolution: {match['Amazon Response']}\n\n"

    prompt = f"""You are an expert customer support agent for Amazon. Use the following past resolved customer support cases as reference context to construct a helpful, polite, and accurate response to the user's issue.

Reference Context:
{context_str}
Current Customer Issue: {user_query}

Draft a clear and direct customer support response:"""
    return prompt

# 2. Test RAG Prompt Construction
query = "My package says delivered but I never received it"
matches = retrieve_similar_qa(query, k=2)
rag_prompt = generate_rag_prompt(query, matches)

print("--- GENERATED RAG PROMPT ---")
print(rag_prompt)

--- GENERATED RAG PROMPT ---
You are an expert customer support agent for Amazon. Use the following past resolved customer support cases as reference context to construct a helpful, polite, and accurate response to the user's issue.

Reference Context:
Context 1:
Customer Query: It said delivered but I haven't gotten it,
Brand Resolution: Oh no! I'm so sorry your package did not arrive! Check out the following Help page for more insight on packages scanned as delivered, https://t.co/cRngkypMPA. ^AV

Context 2:
Customer Query: Had a package scheduled for delivery today and it didn’t come I’m not happy
Brand Resolution: I'm sorry your shipment hasn't arrived; what is the latest status on the order: https://t.co/Y5jpI9gRhE? ^LR


Current Customer Issue: My package says delivered but I never received it

Draft a clear and direct customer support response:


In [ ]:
# 1. Install evaluation dependencies
!pip install -q rouge-score evaluate

import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 2. Load Flan-T5 explicitly using Auto classes
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 3. Tokenize input prompt and generate response
inputs = tokenizer(rag_prompt, return_tensors="pt", max_length=512, truncation=True)
outputs = model.generate(**inputs, max_new_tokens=100)
generated_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- RAG GENERATED RESPONSE ---")
print(generated_output)

# 4. Evaluate Generation Quality against Ground Truth
ground_truth_response = matches[0]['Amazon Response']

rouge = evaluate.load('rouge')
results = rouge.compute(predictions=[generated_output], references=[ground_truth_response])

print("\n--- EVALUATION METRICS (ROUGE Score) ---")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

--- RAG GENERATED RESPONSE ---
Thank you for your patience.



--- EVALUATION METRICS (ROUGE Score) ---
rouge1: 0.1143
rouge2: 0.0000
rougeL: 0.0571
rougeLsum: 0.0571


In [ ]:
import numpy as np

# 1. Select a test set of 50 samples from our dataset
eval_sample = english_sample.head(50)

generated_responses = []
ground_truths = []

print("Running batch evaluation on 50 samples...")

for idx, row in eval_sample.iterrows():
    customer_q = row['clean_customer']
    actual_ans = row['clean_brand']

    # Retrieve top 2 matches excluding the exact same query if matched
    retrieved = retrieve_similar_qa(customer_q, k=3)
    # Filter out exact match if present to avoid leaking direct answer
    context_matches = [m for m in retrieved if m['Past Customer Query'] != customer_q][:2]

    # Generate RAG Prompt
    prompt = generate_rag_prompt(customer_q, context_matches)

    # Generate Response
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = model.generate(**inputs, max_new_tokens=100)
    response_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    generated_responses.append(response_text)
    ground_truths.append(actual_ans)

# 2. Compute Batch ROUGE Metrics
batch_rouge = rouge.compute(predictions=generated_responses, references=ground_truths)

print("\n==========================================")
print("     BATCH EVALUATION METRICS (N=50)      ")
print("==========================================")
for metric, score in batch_rouge.items():
    print(f"Average {metric.upper()}: {score:.4f}")where i nee

Running batch evaluation on 50 samples...

     BATCH EVALUATION METRICS (N=50)      
Average ROUGE1: 0.1309
Average ROUGE2: 0.0354
Average ROUGEL: 0.1094
Average ROUGELSUM: 0.1088


In [ ]:
import pandas as pd

df = pd.read_csv("twcs/twcs.csv")
print("Total rows:", len(df))
print("Columns:", list(df.columns))

Total rows: 2811774
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


In [ ]:
# 1. Download dataset (if not already downloaded)
!kaggle datasets download -d thoughtvector/customer-support-on-twitter

# 2. Unzip the file directly into Colab
!unzip -q customer-support-on-twitter.zip -d twcs_data

Dataset URL: https://www.kaggle.com/datasets/thoughtvector/customer-support-on-twitter
License(s): CC-BY-NC-SA-4.0
100% 169M/169M [00:01<00:00, 118MB/s]



In [ ]:
import pandas as pd

# 1. Unzip and automatically overwrite existing files (-o flag)
!unzip -o -q customer-support-on-twitter.zip -d twcs_data

# 2. Load twcs.csv
df = pd.read_csv("twcs_data/twcs.csv")

print(f"Dataset successfully loaded! Total rows: {len(df):,}")
print("Columns:", list(df.columns))
print("\nFirst 3 rows:")
print(df.head(3))

FileNotFoundError: [Errno 2] No such file or directory: 'twcs_data/twcs.csv'

In [ ]:
import os
import pandas as pd

# Check files in current environment
print("Files in working directory:", os.listdir())
if os.path.exists("twcs"):
    print("Files inside 'twcs':", os.listdir("twcs"))

# Find the file path dynamically and load it
if os.path.exists("twcs/twcs.csv"):
    file_path = "twcs/twcs.csv"
elif os.path.exists("twcs.csv"):
    file_path = "twcs.csv"
elif os.path.exists("twcs_data/twcs.csv"):
    file_path = "twcs_data/twcs.csv"
else:
    file_path = None

if file_path:
    df = pd.read_csv(file_path)
    print(f"\nSuccessfully loaded from '{file_path}'! Total rows: {len(df):,}")
    print(df.head(3))
else:
    print("File not found yet.")

Files in working directory: ['.config', 'twcs_data', 'customer-support-on-twitter.zip', 'sample_data']
File not found yet.


In [ ]:
import os
import pandas as pd

# Check contents of twcs_data
print("Contents of twcs_data:", os.listdir("twcs_data"))

# Find any csv file inside twcs_data or subdirectories
csv_files = []
for root, dirs, files in os.walk("twcs_data"):
    for file in files:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

print("Found CSV files:", csv_files)

if csv_files:
    df = pd.read_csv(csv_files[0])
    print(f"\nSuccessfully loaded '{csv_files[0]}'! Total rows: {len(df):,}")
    print(df.head(3))

Contents of twcs_data: ['sample.csv', 'twcs']
Found CSV files: ['twcs_data/sample.csv', 'twcs_data/twcs/twcs.csv']

Successfully loaded 'twcs_data/sample.csv'! Total rows: 93
   tweet_id     author_id  inbound                      created_at  \
0    119237        105834     True  Wed Oct 11 06:55:44 +0000 2017   
1    119238  ChaseSupport    False  Wed Oct 11 13:25:49 +0000 2017   
2    119239        105835     True  Wed Oct 11 13:00:09 +0000 2017   

                                                text response_tweet_id  \
0  @AppleSupport causing the reply to be disregar...            119236   
1  @105835 Your business means a lot to us. Pleas...               NaN   
2  @76328 I really hope you all change but I'm su...            119238   

   in_response_to_tweet_id  
0                      NaN  
1                 119239.0  
2                      NaN  


In [ ]:
import pandas as pd

# Load the full twcs.csv dataset
file_path = "twcs_data/twcs/twcs.csv"
df = pd.read_csv(file_path)

print(f"Successfully loaded '{file_path}'!")
print(f"Total rows: {len(df):,}")
print("Columns:", list(df.columns))
print("\nFirst 3 rows:")
print(df.head(3))

KeyboardInterrupt: 

In [ ]:
import pandas as pd

file_path = "twcs_data/twcs/twcs.csv"

# Load a subset of 200k rows for fast execution
df = pd.read_csv(file_path, nrows=200000)

print(f"Successfully loaded '{file_path}'!")
print(f"Total rows loaded: {len(df):,}")
print("Columns:", list(df.columns))
print("\nFirst 3 rows:")
print(df.head(3))

Successfully loaded 'twcs_data/twcs/twcs.csv'!
Total rows loaded: 200,000
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']

First 3 rows:
   tweet_id   author_id  inbound                      created_at  \
0         1  sprintcare    False  Tue Oct 31 22:10:47 +0000 2017   
1         2      115712     True  Tue Oct 31 22:11:45 +0000 2017   
2         3      115712     True  Tue Oct 31 22:08:27 +0000 2017   

                                                text response_tweet_id  \
0  @115712 I understand. I would like to assist y...                 2   
1      @sprintcare and how do you propose we do that               NaN   
2  @sprintcare I have sent several private messag...                 1   

   in_response_to_tweet_id  
0                      3.0  
1                      1.0  
2                      4.0  


In [ ]:
# 1. Separate customer tweets (inbound) and support responses (outbound)
inbound_df = df[df['inbound'] == True][['tweet_id', 'author_id', 'text', 'response_tweet_id']].copy()
outbound_df = df[df['inbound'] == False][['tweet_id', 'author_id', 'text', 'in_response_to_tweet_id']].copy()

# Rename columns for clarity
inbound_df.columns = ['customer_tweet_id', 'customer_id', 'customer_text', 'response_tweet_id']
outbound_df.columns = ['brand_tweet_id', 'brand_id', 'brand_response_text', 'customer_tweet_id']

# Ensure tweet IDs match data types for joining
outbound_df['customer_tweet_id'] = pd.to_numeric(outbound_df['customer_tweet_id'], errors='coerce')
inbound_df['customer_tweet_id'] = pd.to_numeric(inbound_df['customer_tweet_id'], errors='coerce')

# 2. Merge customer queries with support replies
qa_pairs = pd.merge(inbound_df, outbound_df, on='customer_tweet_id', how='inner')

print(f"Successfully constructed {len(qa_pairs):,} QA Pairs!\n")
print("--- SAMPLE QA PAIRS ---")
for idx, row in qa_pairs.head(3).iterrows():
    print(f"Customer ({row['customer_id']}): {row['customer_text']}")
    print(f"Brand ({row['brand_id']}): {row['brand_response_text']}")
    print("-" * 60)

Successfully constructed 89,112 QA Pairs!

--- SAMPLE QA PAIRS ---
Customer (115712): @sprintcare I have sent several private messages and no one is responding as usual
Brand (sprintcare): @115712 I understand. I would like to assist you. We would need to get you into a private secured link to further assist.
------------------------------------------------------------
Customer (115712): @sprintcare I did.
Brand (sprintcare): @115712 Please send us a Private Message so that we can further assist you. Just click ‘Message’ at the top of your profile.
------------------------------------------------------------
Customer (115712): @sprintcare is the worst customer service
Brand (sprintcare): @115712 Can you please send us a private message, so that I can gain further details about your account?
------------------------------------------------------------


In [ ]:
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer

# 1. Filter for brand and clean text
brand_df = qa_pairs[qa_pairs['brand_id'] == 'AmazonHelp'].copy().reset_index(drop=True)
brand_df['clean_customer_text'] = brand_df['customer_text'].str.replace(r'@\w+', '', regex=True).str.strip()
brand_df['clean_brand_text'] = brand_df['brand_response_text'].str.replace(r'@\w+', '', regex=True).str.strip()

corpus_df = brand_df.head(2000).copy()

# 2. Build FAISS Index
embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(corpus_df['clean_customer_text'].tolist(), show_progress_bar=False)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))
print(f"FAISS index built with {index.ntotal} vectors!")

# 3. Load Flan-T5 Model & Tokenizer Directly
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_rag_response(query, k=3):
    query_vec = embedder.encode([query])
    distances, indices = index.search(np.array(query_vec).astype('float32'), k)

    retrieved_contexts = [corpus_df.iloc[i]['clean_brand_text'] for i in indices[0]]
    context_str = " ".join(retrieved_contexts)

    prompt = f"Answer the customer question based on context.\nContext: {context_str}\nQuestion: {query}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=100)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 4. Evaluate ROUGE on 50 Sample Cases
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
r1_list, r2_list, rl_list = [], [], []

eval_df = corpus_df.head(50)
print("\nEvaluating RAG Pipeline on 50 test queries...")
for idx, row in eval_df.iterrows():
    pred = generate_rag_response(row['clean_customer_text'])
    ref = row['clean_brand_text']

    scores = scorer.score(ref, pred)
    r1_list.append(scores['rouge1'].fmeasure)
    r2_list.append(scores['rouge2'].fmeasure)
    rl_list.append(scores['rougeL'].fmeasure)

print("\n--- FINAL AGGREGATE EVALUATION SCORES (ROUGE) ---")
print(f"ROUGE-1: {np.mean(r1_list):.4f}")
print(f"ROUGE-2: {np.mean(r2_list):.4f}")
print(f"ROUGE-L: {np.mean(rl_list):.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index built with 2000 vectors!


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


Evaluating RAG Pipeline on 50 test queries...


In [18]:
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer

# 1. Filter and prepare data
brand_df = qa_pairs[qa_pairs['brand_id'] == 'AmazonHelp'].copy().reset_index(drop=True)
brand_df['clean_customer_text'] = brand_df['customer_text'].str.replace(r'@\w+', '', regex=True).str.strip()
brand_df['clean_brand_text'] = brand_df['brand_response_text'].str.replace(r'@\w+', '', regex=True).str.strip()

corpus_df = brand_df.head(2000).copy()

# 2. Build FAISS Index
embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(corpus_df['clean_customer_text'].tolist(), show_progress_bar=False)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

# 3. Load Flan-T5
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Agent Function: Intent + Reply + Escalation
def generate_rag_agent_response(query, k=3):
    query_vec = embedder.encode([query])
    distances, indices = index.search(np.array(query_vec).astype('float32'), k)

    retrieved_contexts = [corpus_df.iloc[i]['clean_brand_text'] for i in indices[0]]
    context_str = " ".join(retrieved_contexts)

    prompt = f"Context: {context_str}\nQuestion: {query}\nProvide Intent, Reply, and Escalation Decision:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=150)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Baselines
def baseline_trivial(query):
    return "Please send us a DM with your order ID so we can investigate."

def baseline_simple_llm(query):
    prompt = f"Reply to this support query: {query}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=128)
    outputs = model.generate(**inputs, max_new_tokens=60)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Evaluation Loop
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
eval_df = corpus_df.head(150)

rag_r1, b1_r1, b2_r1 = [], [], []

print("Evaluating RAG System and Baselines on 150 test queries...")
for idx, row in eval_df.iterrows():
    query = row['clean_customer_text']
    ref = row['clean_brand_text']

    rag_pred = generate_rag_agent_response(query)
    rag_r1.append(scorer.score(ref, rag_pred)['rouge1'].fmeasure)

    b1_pred = baseline_trivial(query)
    b1_r1.append(scorer.score(ref, b1_pred)['rouge1'].fmeasure)

    b2_pred = baseline_simple_llm(query)
    b2_r1.append(scorer.score(ref, b2_pred)['rouge1'].fmeasure)

print("\n--- FINAL EVALUATION RESULTS ---")
print(f"Trivial Baseline ROUGE-1: {np.mean(b1_r1):.4f}")
print(f"Simple LLM Baseline ROUGE-1: {np.mean(b2_r1):.4f}")
print(f"Our RAG Agent ROUGE-1: {np.mean(rag_r1):.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Evaluating RAG System and Baselines on 150 test queries...

--- FINAL EVALUATION RESULTS ---
Trivial Baseline ROUGE-1: 0.1246
Simple LLM Baseline ROUGE-1: 0.1009
Our RAG Agent ROUGE-1: 0.1464
